### Import Requirements 

In [8]:
import os
import glob
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.integrate import simpson

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import joblib

mne.set_log_level("WARNING")

### dataset path 

In [5]:
DATA_ROOT = r"C:\Users\Dell\Desktop\EEG_detection\data\ASZED"
OUTPUT_DIR = r"./data/processed_aszed"

os.makedirs(OUTPUT_DIR, exist_ok=True)

### Channel Standardization

In [27]:
TARGET_CHANNELS = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6","O1","O2"
]


def standardize_channels(raw):

    raw.rename_channels(lambda x: x.strip())

    # Keep only EEG channels
    raw.pick_types(eeg=True)

    # Add missing channels as zeros
    missing = [ch for ch in TARGET_CHANNELS if ch not in raw.ch_names]

    if missing:
        info = mne.create_info(missing, raw.info["sfreq"], ch_types="eeg")
        zeros = np.zeros((len(missing), raw.n_times))
        raw_missing = mne.io.RawArray(zeros, info)
        raw.add_channels([raw_missing], force_update_info=True)

    # Now pick in fixed order
    raw.pick_channels(TARGET_CHANNELS)

    raw.set_montage("standard_1020", on_missing="ignore")

    return raw

### Load EEG File

In [28]:
def load_eeg(filepath):

    raw = mne.io.read_raw_edf(filepath, preload=True, verbose=False)

    raw = standardize_channels(raw)

    # Resample
    raw.resample(250)

    # Notch + bandpass
    raw.notch_filter(50)
    raw.filter(0.5, 45)

    # Re-reference
    raw.set_eeg_reference('average')

    return raw

### Create Epochs

In [29]:
def create_epochs(raw, duration=1.0, overlap=0.5):

    events = mne.make_fixed_length_events(
        raw,
        duration=duration,
        overlap=overlap
    )

    epochs = mne.Epochs(
        raw,
        events,
        tmin=0,
        tmax=duration,
        baseline=None,
        preload=True,
        verbose=False
    )

    return epochs

### Feature Extraction (PSD Bands)

In [30]:
BANDS = {
    "delta": (0.5,4),
    "theta": (4,8),
    "alpha": (8,13),
    "beta": (13,30),
    "gamma": (30,45)
}


def extract_features(epochs):

    psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
    psds = psd.get_data()
    freqs = psd.freqs

    n_epochs, n_channels, _ = psds.shape

    features = []

    for ep in range(n_epochs):

        ep_feat = []

        for ch in range(n_channels):

            signal = psds[ep, ch]

            total_power = simpson(signal, freqs)

            for band in BANDS.values():

                fmin, fmax = band
                mask = (freqs >= fmin) & (freqs <= fmax)

                band_power = simpson(signal[mask], freqs[mask])

                rel_power = band_power / (total_power + 1e-10)

                ep_feat.extend([band_power, rel_power])

        features.append(ep_feat)

    features = np.array(features)

    return features.mean(axis=0)

### Find EDF Files

In [31]:
def find_edf_files(root):

    files = []

    for path, _, f in os.walk(root):
        for file in f:
            if file.lower().endswith(".edf"):
                files.append(os.path.join(path, file))

    return files


edf_files = find_edf_files(DATA_ROOT)

print("Total EDF files:", len(edf_files))

Total EDF files: 1932


### Labels (Simple Example)

In [32]:
def get_label(filepath):

    name = filepath.lower()

    if "control" in name or "healthy" in name:
        return 0
    else:
        return 1

 
 ### Process Dataset

In [33]:
X = []
y = []

for file in tqdm(edf_files):

    try:
        raw = load_eeg(file)
        epochs = create_epochs(raw)

        feats = extract_features(epochs)

        label = get_label(file)

        X.append(feats)
        y.append(label)

    except Exception as e:
        print("Error:", file, e)


X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)

  0%|          | 0/1932 [00:00<?, ?it/s]C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\1113091117.py:11: RuntimeWarning: filter_length (1651) is longer than the signal (1250), distortion is likely. Reduce filter length or filter a longer signal.
  raw.notch_filter(50)
C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\1113091117.py:12: RuntimeWarning: filter_length (1651) is longer than the signal (1250), distortion is likely. Reduce filter length or filter a longer signal.
  raw.filter(0.5, 45)
C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\3614913489.py:12: UserWarning: Zero value in spectrum for channels Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2
  psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
  0%|          | 1/1932 [00:00<03:40,  8.74it/s]C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\3614913489.py:12: UserWarning: Zero value in spectrum for channels Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2
  psd =

Error: C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_3\subject_145\2\Phase 5.edf epochs._get_data() can't run because this Epochs-object is empty. You might want to check Epochs.drop_log or Epochs.plot_drop_log() to see why epochs were dropped.


 97%|█████████▋| 1871/1932 [09:13<00:08,  7.04it/s]C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\3614913489.py:12: UserWarning: Zero value in spectrum for channels Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2
  psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
 97%|█████████▋| 1872/1932 [09:13<00:08,  6.94it/s]C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\3614913489.py:12: UserWarning: Zero value in spectrum for channels Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2
  psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
 97%|█████████▋| 1873/1932 [09:14<00:20,  2.82it/s]C:\Users\Dell\AppData\Local\Temp\ipykernel_6456\3614913489.py:12: UserWarning: Zero value in spectrum for channels Fp1, Fp2, F7, F3, Fz, F4, F8, T3, C3, Cz, C4, T4, T5, P3, Pz, P4, T6, O1, O2
  psd = epochs.compute_psd(method="welch", fmin=0.5, fmax=45)
 97%|█████████▋| 1874/1932 [09:14<00:17,  3.39it/s]C:\Users\Dell\AppData\Local\Temp\i

Dataset shape: (1931, 190)


### Train Test split

In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [35]:
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1544, 190))

In [36]:
model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

In [37]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=cv)

print("CV Accuracy:", scores)
print("Mean:", scores.mean())

CV Accuracy: [1. 1. 1. 1. 1.]
Mean: 1.0


In [40]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

IndexError: index 1 is out of bounds for axis 1 with size 1

In [41]:
print("Unique labels in y:", np.unique(y))
print("Unique labels in y_train:", np.unique(y_train))
print("Model classes:", model.classes_)

Unique labels in y: [1]
Unique labels in y_train: [1]
Model classes: [1]


In [45]:
for f in edf_files[:10]:
    print(f)
    print("  parts:", f.replace("\\", "/").split("/"))
    print()

C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 1.edf
  parts: ['C:', 'Users', 'Dell', 'Desktop', 'EEG_detection', 'data', 'ASZED', 'version_1.1', 'node_1', 'subset_1', 'subject_10', '1', 'Phase 1.edf']

C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 2.edf
  parts: ['C:', 'Users', 'Dell', 'Desktop', 'EEG_detection', 'data', 'ASZED', 'version_1.1', 'node_1', 'subset_1', 'subject_10', '1', 'Phase 2.edf']

C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 3.edf
  parts: ['C:', 'Users', 'Dell', 'Desktop', 'EEG_detection', 'data', 'ASZED', 'version_1.1', 'node_1', 'subset_1', 'subject_10', '1', 'Phase 3.edf']

C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 4.edf
  parts: ['C:', 'Users', 'Dell', 'Desktop', 'EEG_detection', 'data', 'ASZED', 'version_1.1', 'node_1', 'subset_1', 'subject_10', '1', 'Phase 4.edf']



In [43]:
for f in edf_files[:20]:
    print(f)

C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 1.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 2.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 3.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\1\Phase 4.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\2\Phase 1.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\2\Phase 2.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\2\Phase 3.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_10\2\Phase 4.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_11\1\Phase 1.edf
C:\Users\Dell\Desktop\EEG_detection\data\ASZED\version_1.1\node_1\subset_1\subject_11\1\Phase 2.edf


In [46]:
print(np.unique(y))  # must show [0, 1], not just [1]
print(pd.Series(y).value_counts())  # check class balance

[1]
1    1931
Name: count, dtype: int64


In [47]:
raw.notch_filter(50, method='iir')
raw.filter(0.5, 45, method='iir')

<RawEDF | Phase 5.edf, 19 x 1000 (4.0 s), ~176 KiB, data loaded>

In [48]:
y_pred = model.predict(X_test)

if len(model.classes_) == 2:
    y_prob = model.predict_proba(X_test)[:, 1]
    print("ROC AUC:", roc_auc_score(y_test, y_prob))
else:
    print("WARNING: Only one class in training data!")

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 1.0
              precision    recall  f1-score   support

           1       1.00      1.00      1.00       387

    accuracy                           1.00       387
   macro avg       1.00      1.00      1.00       387
weighted avg       1.00      1.00      1.00       387

